In [35]:
import os
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import cv2
import glob
from tqdm import tqdm
from torchvision.models import vgg19
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import warnings
warnings.filterwarnings('ignore')

In [15]:
SCALE_FACTOR = 4
IMG_SIZE = 128 
BATCH_SIZE = 16  
EPOCHS = 100
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
KAGGLE_OUTPUT = '/kaggle/working/'
MAX_IMAGES = 500 
PATCHES_PER_IMAGE = 4  

In [16]:
class DIV2KDataPrep:
    def __init__(self, hr_train_dir, hr_test_dir, output_path):
        self.hr_train_dir = hr_train_dir
        self.hr_test_dir = hr_test_dir
        self.output_path = output_path
        
    def bicubic_downsample(self, hr_img, scale_factor):
        h, w = hr_img.shape[:2]
        lr_h, lr_w = h // scale_factor, w // scale_factor
        lr_img = cv2.resize(hr_img, (lr_w, lr_h), interpolation=cv2.INTER_CUBIC)
        return lr_img
    
    def prepare_dataset(self):
        hr_train_images = sorted(glob.glob(os.path.join(self.hr_train_dir, '*.png')))[:MAX_IMAGES//2]
        hr_test_images = sorted(glob.glob(os.path.join(self.hr_test_dir, '*.png')))[:MAX_IMAGES//2]
        
        all_hr_images = hr_train_images + hr_test_images
        print(f"Processing {len(all_hr_images)} images...")
        
        lr_list, hr_list = [], []
        patch_count = 0
        
        for hr_path in tqdm(all_hr_images, desc="Processing images"):
            hr_img = cv2.imread(hr_path)
            if hr_img is None:
                continue
            
            # Create limited patches from HR image
            h, w = hr_img.shape[:2]
            patch_idx = 0
            for i in range(0, h - IMG_SIZE, IMG_SIZE):
                for j in range(0, w - IMG_SIZE, IMG_SIZE):
                    if patch_idx >= PATCHES_PER_IMAGE:
                        break
                    
                    hr_patch = hr_img[i:i+IMG_SIZE, j:j+IMG_SIZE]
                    if hr_patch.shape != (IMG_SIZE, IMG_SIZE, 3):
                        continue
                    
                    lr_patch = self.bicubic_downsample(hr_patch, SCALE_FACTOR)
                    lr_list.append(lr_patch)
                    hr_list.append(hr_patch)
                    patch_idx += 1
                    patch_count += 1
        
        print(f"Total patches extracted: {patch_count}")
        
        lr_array = np.array(lr_list, dtype=np.float32) / 255.0
        hr_array = np.array(hr_list, dtype=np.float32) / 255.0
        
        # Split: 80% train, 10% val, 10% test
        idx = np.arange(len(lr_array))
        train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=42)
        val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)
        
        with h5py.File(self.output_path, 'w') as f:
            f.create_dataset('lr_train', data=lr_array[train_idx], compression='gzip')
            f.create_dataset('hr_train', data=hr_array[train_idx], compression='gzip')
            f.create_dataset('lr_val', data=lr_array[val_idx], compression='gzip')
            f.create_dataset('hr_val', data=hr_array[val_idx], compression='gzip')
            f.create_dataset('lr_test', data=lr_array[test_idx], compression='gzip')
            f.create_dataset('hr_test', data=hr_array[test_idx], compression='gzip')
        
        print(f"Dataset saved: {self.output_path}")
        print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
        return (len(train_idx), len(val_idx), len(test_idx))

In [17]:
class SuperResolutionDataset(Dataset):
    def __init__(self, h5_path, split='train'):
        self.h5_path = h5_path
        self.split = split
        with h5py.File(h5_path, 'r') as f:
            self.lr_data = f[f'lr_{split}'][:]
            self.hr_data = f[f'hr_{split}'][:]
    
    def __len__(self):
        return len(self.lr_data)
    
    def __getitem__(self, idx):
        lr = torch.from_numpy(self.lr_data[idx]).permute(2, 0, 1)
        hr = torch.from_numpy(self.hr_data[idx]).permute(2, 0, 1)
        return lr, hr

In [18]:
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return x

In [19]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        x = self.relu(self.conv1(x))
        x = self.conv2(x)
        return x + residual

In [20]:
class SRGANGenerator(nn.Module):
    def __init__(self, num_residual_blocks=16):
        super(SRGANGenerator, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 9, padding=4)
        self.residual_blocks = nn.Sequential(*[ResidualBlock(64) for _ in range(num_residual_blocks)])
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
        # No upsampling since input is already upsampled
        self.conv3 = nn.Conv2d(64, 3, 9, padding=4)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        residual = x
        x = self.residual_blocks(x)
        x = self.conv2(x)
        x = x + residual
        x = self.conv3(x)
        return x

In [21]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(128, 1, 1)
        )
    
    def forward(self, x):
        return self.net(x)

In [22]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True)
        self.features = nn.Sequential(*list(vgg.features.children())[:36])
        for param in self.features.parameters():
            param.requires_grad = False
    
    def forward(self, x, y):
        return torch.mean((self.features(x) - self.features(y)) ** 2)

In [23]:
def train_srcnn(model, train_loader, val_loader, epochs=100):
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    
    train_loss, val_loss = [], []
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0
        for lr, hr in tqdm(train_loader, desc=f"SRCNN Epoch {epoch+1}/{epochs}"):
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            
            # Upsample LR to HR size using bicubic
            lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
            sr = model(lr_up)
            
            loss = criterion(sr, hr)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        train_loss.append(avg_train_loss)
        
        # Validate
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for lr, hr in val_loader:
                lr, hr = lr.to(DEVICE), hr.to(DEVICE)
                lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
                sr = model(lr_up)
                loss = criterion(sr, hr)
                running_val_loss += loss.item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_loss.append(avg_val_loss)
        scheduler.step()
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srcnn_best.pt'))
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")
    
    torch.save(model.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srcnn_final.pt'))
    return train_loss, val_loss

In [24]:
def train_srgan(generator, discriminator, train_loader, val_loader, epochs=100):
    g_optimizer = optim.Adam(generator.parameters(), lr=LR, betas=(0.9, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.9, 0.999))
    
    criterion_mse = nn.MSELoss()
    criterion_bce = nn.BCEWithLogitsLoss()
    perceptual_loss = PerceptualLoss().to(DEVICE)
    
    g_train_loss, d_train_loss, val_loss = [], [], []
    best_val_loss = float('inf')
    
    # Pretrain generator with MSE
    print("\n=== Pretraining Generator with MSE ===")
    for epoch in range(20):
        generator.train()
        running_loss = 0.0
        for lr, hr in tqdm(train_loader, desc=f"Pretrain Epoch {epoch+1}/20"):
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
            sr = generator(lr_up)
            
            loss = criterion_mse(sr, hr)
            g_optimizer.zero_grad()
            loss.backward()
            g_optimizer.step()
            running_loss += loss.item()
        
        if (epoch + 1) % 5 == 0:
            print(f"Pretrain Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}")
    
    torch.save(generator.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srgan_generator_pretrained.pt'))
    
    # Adversarial training
    print("\n=== Adversarial Training ===")
    for epoch in range(epochs):
        generator.train()
        discriminator.train()
        
        running_g_loss = 0.0
        running_d_loss = 0.0
        
        for lr, hr in tqdm(train_loader, desc=f"SRGAN Epoch {epoch+1}/{epochs}"):
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
            
            batch_size = lr.size(0)
            real_label = torch.ones(batch_size, 1, 1, 1).to(DEVICE)
            fake_label = torch.zeros(batch_size, 1, 1, 1).to(DEVICE)
            
            # Train Discriminator
            sr = generator(lr_up).detach()
            
            d_real = discriminator(hr)
            d_fake = discriminator(sr)
            
            d_loss_real = criterion_bce(d_real, real_label)
            d_loss_fake = criterion_bce(d_fake, fake_label)
            d_loss = d_loss_real + d_loss_fake
            
            d_optimizer.zero_grad()
            d_loss.backward()
            d_optimizer.step()
            running_d_loss += d_loss.item()
            
            # Train Generator
            sr = generator(lr_up)
            d_fake = discriminator(sr)
            
            g_loss_adversarial = criterion_bce(d_fake, real_label)
            g_loss_perceptual = perceptual_loss(sr, hr)
            g_loss_mse = criterion_mse(sr, hr)
            
            g_loss = 0.001 * g_loss_adversarial + 0.006 * g_loss_perceptual + g_loss_mse
            
            g_optimizer.zero_grad()
            g_loss.backward()
            g_optimizer.step()
            running_g_loss += g_loss.item()
        
        avg_g_loss = running_g_loss / len(train_loader)
        avg_d_loss = running_d_loss / len(train_loader)
        g_train_loss.append(avg_g_loss)
        d_train_loss.append(avg_d_loss)
        
        # Validate
        generator.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for lr, hr in val_loader:
                lr, hr = lr.to(DEVICE), hr.to(DEVICE)
                lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
                sr = generator(lr_up)
                loss = criterion_mse(sr, hr)
                running_val_loss += loss.item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_loss.append(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(generator.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srgan_generator_best.pt'))
            torch.save(discriminator.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srgan_discriminator_best.pt'))
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: G Loss={avg_g_loss:.4f}, D Loss={avg_d_loss:.4f}, Val Loss={avg_val_loss:.4f}")
    
    torch.save(generator.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srgan_generator_final.pt'))
    torch.save(discriminator.state_dict(), os.path.join(KAGGLE_OUTPUT, 'srgan_discriminator_final.pt'))
    return g_train_loss, d_train_loss, val_loss

In [25]:
if __name__ == '__main__':
    # Step 1: Prepare data (only if not already done)
    h5_path = os.path.join(KAGGLE_OUTPUT, 'div2k_dataset.h5')
    hr_train_dir = '/kaggle/input/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR'
    hr_test_dir = '/kaggle/input/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR'  # Same as train since no test
    
    if not os.path.exists(h5_path):
        print("Preparing dataset...")
        if os.path.exists(hr_train_dir):
            prep = DIV2KDataPrep(hr_train_dir, hr_test_dir, h5_path)
            prep.prepare_dataset()
            print("Dataset preparation complete!")
        else:
            print(f"Error: Directory not found: {hr_train_dir}")
    else:
        print("Dataset already exists. Loading...")
    
    # Step 2: Create data loaders
    train_dataset = SuperResolutionDataset(h5_path, 'train')
    val_dataset = SuperResolutionDataset(h5_path, 'val')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Step 3: Train SRCNN
    print("\n=== Training SRCNN ===")
    srcnn = SRCNN().to(DEVICE)
    train_loss, val_loss = train_srcnn(srcnn, train_loader, val_loader, epochs=50)
    
    # Plot metrics
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, label='Train Loss')
    plt.plot(val_loss, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('SRCNN Training')
    plt.legend()
    plt.savefig(os.path.join(KAGGLE_OUTPUT, 'srcnn_training.png'))
    plt.close()
    
    # Step 4: Train SRGAN
    print("\n=== Training SRGAN ===")
    generator = SRGANGenerator().to(DEVICE)
    discriminator = Discriminator().to(DEVICE)
    g_loss, d_loss, val_loss = train_srgan(generator, discriminator, train_loader, val_loader, epochs=50)
    
    # Plot SRGAN metrics
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(g_loss, label='Generator Loss')
    plt.plot(d_loss, label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('SRGAN Adversarial Training')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(val_loss, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('SRGAN Validation')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(KAGGLE_OUTPUT, 'srgan_training.png'))
    plt.close()
    
    print("\nTraining complete! All models saved to /kaggle/working/")

Preparing dataset...
Processing 500 images...


Processing images: 100%|██████████| 500/500 [00:50<00:00,  9.91it/s]


Total patches extracted: 2000
Dataset saved: /kaggle/working/div2k_dataset.h5
Train: 1600, Val: 200, Test: 200
Dataset preparation complete!

=== Training SRCNN ===


SRCNN Epoch 10/50: 100%|██████████| 100/100 [00:06<00:00, 16.16it/s]


Epoch 10: Train Loss=0.0021, Val Loss=0.0018


...

Epoch 30: G Loss=0.0027, D Loss=1.2987, Val Loss=0.0015


SRGAN Epoch 40/50: 100%|██████████| 100/100 [01:38<00:00,  1.02it/s]


Epoch 40: G Loss=0.0027, D Loss=1.2914, Val Loss=0.0015


SRGAN Epoch 50/50: 100%|██████████| 100/100 [01:38<00:00,  1.02it/s]


Epoch 50: G Loss=0.0026, D Loss=1.3041, Val Loss=0.0016

Training complete! All models saved to /kaggle/working/


In [26]:
!zip -r working_dir.zip /kaggle/working/*

  adding: kaggle/working/div2k_dataset.h5 (deflated 2%)
  adding: kaggle/working/srcnn_best.pt (deflated 8%)
  adding: kaggle/working/srcnn_final.pt (deflated 8%)
  adding: kaggle/working/srcnn_training.png (deflated 15%)
  adding: kaggle/working/srgan_discriminator_best.pt (deflated 8%)
  adding: kaggle/working/srgan_discriminator_final.pt (deflated 7%)
  adding: kaggle/working/srgan_generator_best.pt (deflated 8%)
  adding: kaggle/working/srgan_generator_final.pt (deflated 8%)
  adding: kaggle/working/srgan_generator_pretrained.pt (deflated 8%)
  adding: kaggle/working/srgan_training.png (deflated 8%)


In [37]:
def calculate_psnr(img1, img2):
    """Calculate PSNR between two images (0-1 range)"""
    img1_np = img1.cpu().numpy()
    img2_np = img2.cpu().numpy()
    return peak_signal_noise_ratio(img2_np, img1_np, data_range=1.0)

def calculate_ssim(img1, img2):
    """Calculate SSIM between two images (0-1 range)"""
    img1_np = img1.cpu().numpy()
    img2_np = img2.cpu().numpy()
    return structural_similarity(img2_np, img1_np, data_range=1.0, channel_axis=0)

def evaluate_model(model, test_loader, model_name="Model"):
    """Evaluate model on test set"""
    model.eval()
    psnr_scores = []
    ssim_scores = []
    
    with torch.no_grad():
        for lr, hr in tqdm(test_loader, desc=f"Evaluating {model_name}"):
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            
            # Upsample LR using bicubic
            lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
            sr = model(lr_up)
            
            # Clamp to valid range
            sr = torch.clamp(sr, 0, 1)
            
            for i in range(sr.size(0)):
                psnr = calculate_psnr(sr[i], hr[i])
                ssim = calculate_ssim(sr[i], hr[i])
                psnr_scores.append(psnr)
                ssim_scores.append(ssim)
    
    avg_psnr = np.mean(psnr_scores)
    avg_ssim = np.mean(ssim_scores)
    
    print(f"\n{model_name} Results:")
    print(f"  Average PSNR: {avg_psnr:.2f} dB")
    print(f"  Average SSIM: {avg_ssim:.4f}")
    
    return avg_psnr, avg_ssim, psnr_scores, ssim_scores

def visualize_results(model, test_loader, num_samples=4, model_name="Model"):
    """Visual comparison of LR, SR, and HR images"""
    model.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    sample_count = 0
    with torch.no_grad():
        for lr, hr in test_loader:
            if sample_count >= num_samples:
                break
            
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)
            lr_up = torch.nn.functional.interpolate(lr, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False)
            sr = model(lr_up)
            sr = torch.clamp(sr, 0, 1)
            
            for i in range(lr.size(0)):
                if sample_count >= num_samples:
                    break
                
                # Low-resolution (bicubic upsampled)
                axes[sample_count, 0].imshow(lr_up[i].permute(1, 2, 0).cpu().numpy())
                axes[sample_count, 0].set_title("LR (Bicubic)" if sample_count == 0 else "")
                axes[sample_count, 0].axis('off')
                
                # Super-resolved
                axes[sample_count, 1].imshow(sr[i].permute(1, 2, 0).cpu().numpy())
                axes[sample_count, 1].set_title(f"SR ({model_name})" if sample_count == 0 else "")
                axes[sample_count, 1].axis('off')
                
                # High-resolution ground truth
                axes[sample_count, 2].imshow(hr[i].permute(1, 2, 0).cpu().numpy())
                axes[sample_count, 2].set_title("HR (Ground Truth)" if sample_count == 0 else "")
                axes[sample_count, 2].axis('off')
                
                # Difference map
                diff = torch.abs(sr[i] - hr[i]).mean(dim=0)
                axes[sample_count, 3].imshow(diff.cpu().numpy(), cmap='hot')
                axes[sample_count, 3].set_title("Error Map" if sample_count == 0 else "")
                axes[sample_count, 3].axis('off')
                
                sample_count += 1
    
    plt.tight_layout()
    plt.savefig(os.path.join(KAGGLE_OUTPUT, f'{model_name}_visual_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Visual comparison saved: {model_name}_visual_comparison.png")
class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True)
        self.features = nn.Sequential(*list(vgg.features.children())[:36])
        for param in self.features.parameters():
            param.requires_grad = False
    
    def forward(self, x, y):
        return torch.mean((self.features(x) - self.features(y)) ** 2)

In [38]:
def evaluate_all_models():
    """Standalone evaluation function - run models on test set"""
    h5_path = os.path.join(KAGGLE_OUTPUT, 'div2k_dataset.h5')
    
    if not os.path.exists(h5_path):
        print("Error: Dataset not found. Run training first!")
        return
    
    # Create test loader
    test_dataset = SuperResolutionDataset(h5_path, 'test')
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    print("\n" + "="*50)
    print("EVALUATION ON TEST SET")
    print("="*50)
    
    # Load SRCNN
    print("\nLoading SRCNN model...")
    srcnn = SRCNN().to(DEVICE)
    srcnn_path = os.path.join(KAGGLE_OUTPUT, 'srcnn_best.pt')
    if os.path.exists(srcnn_path):
        srcnn.load_state_dict(torch.load(srcnn_path, map_location=DEVICE))
        srcnn_psnr, srcnn_ssim, _, _ = evaluate_model(srcnn, test_loader, "SRCNN")
        visualize_results(srcnn, test_loader, num_samples=4, model_name="SRCNN")
    else:
        print(f"SRCNN model not found at {srcnn_path}")
        srcnn_psnr, srcnn_ssim = None, None
    
    # Load SRGAN
    print("\nLoading SRGAN model...")
    generator = SRGANGenerator().to(DEVICE)
    srgan_path = os.path.join(KAGGLE_OUTPUT, 'srgan_generator_best.pt')
    if os.path.exists(srgan_path):
        generator.load_state_dict(torch.load(srgan_path, map_location=DEVICE))
        srgan_psnr, srgan_ssim, _, _ = evaluate_model(generator, test_loader, "SRGAN Generator")
        visualize_results(generator, test_loader, num_samples=4, model_name="SRGAN")
    else:
        print(f"SRGAN model not found at {srgan_path}")
        srgan_psnr, srgan_ssim = None, None
    
    # Create comparison
    if srcnn_psnr is not None and srgan_psnr is not None:
        print("\n" + "="*50)
        print("MODEL COMPARISON")
        print("="*50)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        models = ['SRCNN', 'SRGAN']
        psnr_vals = [srcnn_psnr, srgan_psnr]
        ssim_vals = [srcnn_ssim, srgan_ssim]
        
        axes[0].bar(models, psnr_vals, color=['#3498db', '#e74c3c'])
        axes[0].set_ylabel('PSNR (dB)')
        axes[0].set_title('PSNR Comparison')
        axes[0].grid(axis='y', alpha=0.3)
        for i, v in enumerate(psnr_vals):
            axes[0].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')
        
        axes[1].bar(models, ssim_vals, color=['#3498db', '#e74c3c'])
        axes[1].set_ylabel('SSIM')
        axes[1].set_title('SSIM Comparison')
        axes[1].set_ylim([0, 1])
        axes[1].grid(axis='y', alpha=0.3)
        for i, v in enumerate(ssim_vals):
            axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(KAGGLE_OUTPUT, 'model_comparison.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"\n{'Metric':<15} {'SRCNN':<15} {'SRGAN':<15}")
        print("-" * 45)
        print(f"{'PSNR (dB)':<15} {srcnn_psnr:<15.2f} {srgan_psnr:<15.2f}")
        print(f"{'SSIM':<15} {srcnn_ssim:<15.4f} {srgan_ssim:<15.4f}")
        
        # Determine winner
        print("\n" + "="*50)
        if srcnn_psnr > srgan_psnr:
            print(f"SRCNN has better PSNR: {srcnn_psnr:.2f} dB vs {srgan_psnr:.2f} dB")
        else:
            print(f"SRGAN has better PSNR: {srgan_psnr:.2f} dB vs {srcnn_psnr:.2f} dB")
        
        if srcnn_ssim > srgan_ssim:
            print(f"SRCNN has better SSIM: {srcnn_ssim:.4f} vs {srgan_ssim:.4f}")
        else:
            print(f"SRGAN has better SSIM: {srgan_ssim:.4f} vs {srcnn_ssim:.4f}")
        print("="*50)

In [39]:
evaluate_all_models()


EVALUATION ON TEST SET

Loading SRCNN model...


Evaluating SRCNN: 100%|██████████| 13/13 [00:01<00:00, 11.79it/s]



SRCNN Results:
  Average PSNR: 35.91 dB
  Average SSIM: 0.8754
Visual comparison saved: SRCNN_visual_comparison.png

Loading SRGAN model...


Evaluating SRGAN Generator: 100%|██████████| 13/13 [00:02<00:00,  4.36it/s]



SRGAN Generator Results:
  Average PSNR: 36.07 dB
  Average SSIM: 0.8779
Visual comparison saved: SRGAN_visual_comparison.png

MODEL COMPARISON

Metric          SRCNN           SRGAN          
---------------------------------------------
PSNR (dB)       35.91           36.07          
SSIM            0.8754          0.8779         

SRGAN has better PSNR: 36.07 dB vs 35.91 dB
SRGAN has better SSIM: 0.8779 vs 0.8754
